# Data Integration and Aggregation

### Creating a spark session for data interaction

In [1]:
from pyspark.sql import SparkSession


spark = (SparkSession.builder
         .appName('Olist Data Integration & Aggregation')
         .config('spark.executor.memory', '10g')
         .config('spark.executor.cores', '4')
         .config('spark.executor.instances', '2')
         .config('spark.driver.memory', '4g')
         .config('spark.driver.maxResultSize', '2gb')
         .config('spark.sql.shuffle.partitions','64')
         .config('spark.default.parallelism','64')
         .config('spark.sql.adaptive.enabled', True)
         .config('spark.sql.adaptive.coalescePartition.enabled', True)
         .config('spark.sql.adaptive.skewJoin.enabled', True)
         .config('spark.sql.autoBroadcastJoinThreshold', '20mb')
         .config('spark.sql.files.maxPartitionBytes', '64mb')
         .config('spark.sql.files.openCostInBytes', '4mb')
         .config('spark.memory.fraction', 0.8)
         .config('spark.memory.storageFraction',0.2)
         .getOrCreate())

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/29 15:59:07 INFO SparkEnv: Registering MapOutputTracker
26/06/29 15:59:07 INFO SparkEnv: Registering BlockManagerMaster
26/06/29 15:59:07 INFO SparkEnv: Registering BlockManagerMasterHeartbeat
26/06/29 15:59:08 INFO SparkEnv: Registering OutputCommitCoordinator


**_Helper Functions_**

In [2]:
from pyspark.sql.functions import *

def getSchema(df, df_name):
    print(f"Schemas for {df_name.title()}")
    df.printSchema()

### Loading Cleaned Data

In [3]:
hdfs_base_path = '/user/Marho/project/olist/processed/'


customers_df = spark.read.parquet(hdfs_base_path+'customers')
geolocations_df = spark.read.parquet(hdfs_base_path+'geolocations')
order_items_df = spark.read.parquet(hdfs_base_path+'orderItems')
payments_df = spark.read.parquet(hdfs_base_path+'payments')
review_df = spark.read.parquet(hdfs_base_path+'reviews')
orders_df = spark.read.parquet(hdfs_base_path+'orders')
products_df = spark.read.parquet(hdfs_base_path+'products')
sellers_df = spark.read.parquet(hdfs_base_path+'sellers')
products_category_df = spark.read.parquet(hdfs_base_path+'categories_transalation')

## Data Integration

### Joining Datasets - Optimized

In [4]:
# orders -> order_items  (inner — core transaction data)
order_items_joined_df = orders_df.join(order_items_df, 'order_id', 'inner') 

# orders -> products     (inner — product must exist)
order_items_products_df = order_items_joined_df.join(broadcast(products_df), 'product_id', 'inner') # <- optimized (products file size:- 1.3MB)

# orders -> sellers      (inner — seller must exist)
order_items_products_sellers_df = order_items_products_df.join(broadcast(sellers_df), 'seller_id', 'inner')  # <- Optimized (sellers file size:- 128.2k)

# orders -> customers    (inner — customer must exist)
full_orders_with_customers_df  = order_items_products_sellers_df.join(customers_df, 'customer_id', 'inner') 


# orders -> geolocations ( left joins to preserve orders regardless of missing geolocation data)
full_orders_with_geolocation = full_orders_with_customers_df.join(geolocations_df, 
                                                              full_orders_with_customers_df.customer_zip_code_prefix == geolocations_df.geolocation_zip_code_prefix,
                                                             'left')

# orders -> reviews      (left  — preserve orders with missing reviews)
full_orders_with_reviews = full_orders_with_geolocation.join(review_df, 'order_id', 'left')

# orders -> categories   (left  — preserve orders with missing categories)
full_orders_with_categories = full_orders_with_reviews.join(broadcast(products_category_df), 'product_category_name', 'left') # <- Optimized (product category file size :- 3.0KB

# orders -> payments (left — preserve orders regardless of payment status)
full_orders_df = full_orders_with_categories.join(broadcast(payments_df), 'order_id', 'left') # <- optimized (payment file size :- 3.7MB

### Drop Duplicates 

In [5]:
full_orders_df = full_orders_df.dropDuplicates(['customer_unique_id', 'order_id', 'order_purchase_timestamp'])

### Cache complete order details

In [6]:
full_orders_df.cache()

26/06/29 15:59:42 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


DataFrame[order_id: string, product_category_name: string, customer_id: string, seller_id: string, product_id: string, order_status: string, order_purchase_timestamp: timestamp, order_approved_at: timestamp, order_delivered_carrier_date: timestamp, order_delivered_customer_date: timestamp, order_estimated_delivery_date: timestamp, approval_time: int, carrier_handling_time: int, delivery_delay: int, delivery_duration: int, shipping_time: int, purchase_day: int, purchase_day_name: string, purchase_month: int, purchase_month_name: string, purchase_year: int, order_item_id: int, shipping_limit_date: timestamp, price: double, freight_value: double, product_name_lenght: double, product_description_lenght: double, product_photos_qty: double, product_weight_g: double, product_length_cm: double, product_height_cm: double, product_width_cm: double, product_weight_category: string, seller_zip_code_prefix: string, seller_city: string, seller_state: string, customer_unique_id: string, customer_zip_

### Verify Schemas

In [7]:
getSchema(full_orders_df, "Complete Orders")

Schemas for Complete Orders
root
 |-- order_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- approval_time: integer (nullable = true)
 |-- carrier_handling_time: integer (nullable = true)
 |-- delivery_delay: integer (nullable = true)
 |-- delivery_duration: integer (nullable = true)
 |-- shipping_time: integer (nullable = true)
 |-- purchase_day: integer (nullable = true)
 |-- purchase_day_name: string (nullable = true)
 |-- purchase_month: integer (nullable = true)
 |-- purchase_mont

## Data Aggregation

### Total Orders Per Customer

In [8]:
orders_per_customers_df =  (full_orders_df.groupBy('customer_unique_id').agg(countDistinct('order_id').alias('total_order'))
                            .orderBy('total_order', ascending=False))
fmt_orders_per_customers_df = orders_per_customers_df.select('customer_unique_id', format_number('total_order', 0).alias('total_order'))
fmt_orders_per_customers_df.show(10,truncate=False)

+--------------------------------+-----------+
|customer_unique_id              |total_order|
+--------------------------------+-----------+
|8d50f5eadf50201ccdcedfb9e2ac8455|16         |
|3e43e6105506432c953e165fb2acf44c|9          |
|6469f99c1f9dfae7733b25662e7f1782|7          |
|1b6c7548a2a1f9037c1fd3ddfed95f33|7          |
|ca77025e7201e3b30c44b472ff346268|7          |
|47c1a3033b8b77b3ab6e109eb4d5fdf3|6          |
|63cfc61cee11cbe306bff5857d00bfe4|6          |
|f0e310a6839dce9de1638e0fe5ab282a|6          |
|dc813062e0fc23409cd255f7f53c7074|6          |
|b4e4f24de1e8725b74e4a1f4975116ed|5          |
+--------------------------------+-----------+
only showing top 10 rows



### Average Review Score per Seller

In [9]:
seller_avg_review_score_df = (full_orders_df.groupBy('seller_id').agg(avg('review_score').alias('avg_review_score'))
                            .orderBy('avg_review_score', ascending=False))
fmt_seller_avg_review_score_df = seller_avg_review_score_df.select('seller_id', format_number('avg_review_score',0).alias('avg_review_score'))
fmt_seller_avg_review_score_df.show(10,truncate=False)

+--------------------------------+----------------+
|seller_id                       |avg_review_score|
+--------------------------------+----------------+
|edd066cd02126d7800f9b66e980e9931|5               |
|5f5a58930c3c35f3b5af264f34fb8c85|5               |
|1e483cc5c76fef08d3ca05f9a8af7d01|5               |
|39a5005f2605cbdb4f9ac14485cabfd1|5               |
|e5def42655b7490edac5a56fe8e9e603|5               |
|59de1d4c8c057c2a53acd4ca66531af2|5               |
|52b53f7061969fe471d119b6195da864|5               |
|eaaa46e98c9e61f6764dd1d7a2ab8b3e|5               |
|48efc9d94a9834137efd9ea76b065a38|5               |
|404e1ba01358af4cd63f679b2c4d1fa1|5               |
+--------------------------------+----------------+
only showing top 10 rows



### Top 10 Most Sold Products

In [10]:
most_sold_products_df = (full_orders_df.groupBy('product_id').agg(count('order_id').alias('total_order'))
                         .orderBy('total_order', ascending=False))
fmt_most_sold_products_df = most_sold_products_df.select('product_id', format_number('total_order', 0).alias('total_order'))
fmt_most_sold_products_df.show(10)

+--------------------+-----------+
|          product_id|total_order|
+--------------------+-----------+
|99a4788cb24856965...|        438|
|aca2eb7d00ea1a7b8...|        430|
|422879e10f4668299...|        337|
|d1c427060a0f73f6b...|        314|
|53b36df67ebb7c415...|        302|
|389d119b48cf3043d...|        293|
|368c6c730842d7801...|        280|
|53759a2ecddad2bb8...|        275|
|154e7e31ebfa09220...|        267|
|2b4609f8948be1887...|        259|
+--------------------+-----------+
only showing top 10 rows



### Top Customers By Purchase

In [11]:
revenue_customers_df = (full_orders_df.groupBy('customer_id').agg(sum('payment_value').alias('total_amount_spent'))
                        .orderBy('total_amount_spent', ascending=False))
fmt_revenue_customers_df = revenue_customers_df.select('customer_id', format_number('total_amount_spent', 2).alias('total_amount_spent'))
fmt_revenue_customers_df.show(10)

+--------------------+------------------+
|         customer_id|total_amount_spent|
+--------------------+------------------+
|f7622098214b4634b...|          3,242.84|
|71901689c5f3e5adc...|          3,195.73|
|6152fbfc8a92ee25f...|          3,122.72|
|1ff773612ab8934db...|          3,018.60|
|ed583a2a1eaf0dedc...|          2,844.96|
|53e644c7c57416bda...|          2,607.09|
|03a6f3a3935165f5d...|          2,566.90|
|6361b9f3b85d41860...|          2,480.58|
|b9da524b08224eaff...|          2,467.17|
|a1a52dda6ac52aee9...|          2,276.10|
+--------------------+------------------+
only showing top 10 rows



### Sellers Products

In [12]:
seller_products_df = (full_orders_df
                          .groupBy('seller_id', 'product_id', 'price')
                          .agg(
                              count('order_id')
                              .alias('total_orders')
                          )
                          .withColumn(
                              'total_revenue', 
                              col('price') * col('total_orders')
                          )
                     )
(seller_products_df
     .select(
         'seller_id', 'product_id', 
         'price', 'total_orders', 
         format_number('total_revenue', 2).alias('total_revenue')
     )
).show()

+--------------------+--------------------+------+------------+-------------+
|           seller_id|          product_id| price|total_orders|total_revenue|
+--------------------+--------------------+------+------------+-------------+
|7f152321c60a266ed...|1dfb36d969d347f56...|159.99|          11|     1,759.89|
|e24fc9fcd865784fb...|a77aadc991c8f31b9...| 198.0|           1|       198.00|
|3d871de0142ce09b7...|c7944f654db61924b...| 139.0|           3|       417.00|
|f5a590cf36251cf11...|a934477935383c309...| 310.0|           1|       310.00|
|6989574bd97d9773f...|cb1b037c120d9dad4...| 399.9|           1|       399.90|
|b6d44737c04332870...|bf4ff42133c7bced3...| 125.0|           2|       250.00|
|4e922959ae960d389...|3b7929cc079c14493...| 110.0|           4|       440.00|
|a3a38f4affed601eb...|011be89e68e286ac0...|  69.9|           2|       139.80|
|004c9cd9d87a3c30c...|a742e0f6787ec4e2a...|117.99|           2|       235.98|
|1900267e848ceeba8...|ceeb8b2d571b23399...|  53.0|           7| 

## Window and Ranking

### Ranked Top Selling Products Per Seller

In [13]:
from pyspark.sql.window import Window

window_rank_spec = Window.partitionBy('seller_id').orderBy(col('total_orders').desc())

In [14]:
ranked_top_seller_products_df = seller_products_df.withColumn('rank', rank().over(window_rank_spec)).filter(col('rank')<=5)
ranked_top_seller_products_df.select('seller_id', 'product_id', 'price', 'total_orders', 'rank').show(truncate=False)

+--------------------------------+--------------------------------+-----+------------+----+
|seller_id                       |product_id                      |price|total_orders|rank|
+--------------------------------+--------------------------------+-----+------------+----+
|001cca7ae9ae17fb1caed9dfb1094831|08574b074924071f4e201e151b152b4e|99.0 |60          |1   |
|001cca7ae9ae17fb1caed9dfb1094831|08574b074924071f4e201e151b152b4e|89.0 |25          |2   |
|001cca7ae9ae17fb1caed9dfb1094831|e251ebd2858be1aa7d9b2087a6992580|112.0|18          |3   |
|001cca7ae9ae17fb1caed9dfb1094831|0da9ffd92214425d880de3f94e74ce39|112.0|10          |4   |
|001cca7ae9ae17fb1caed9dfb1094831|547b95702aec86f05ac37e61d164891c|129.0|9           |5   |
|001cca7ae9ae17fb1caed9dfb1094831|98a8c2fa16d7239c606640f5555768e4|99.0 |9           |5   |
|001e6ad469a905060d959994f1b41e4f|093cd981b714bcdff182b427d87fc8fc|250.0|1           |1   |
|002100f778ceb8431b7a1020ff7ab48f|158102fe543dbaeb84d87811bfe06d0d|17.9 |13     

### Densed ranked Top revenue generating Products Per Seller

In [15]:
window_drank_spec = Window.partitionBy('seller_id').orderBy(desc('total_revenue'))

dense_ranked_top_seller_products_df  = (seller_products_df
                                        .withColumn(
                                            'dense_rank', 
                                             dense_rank().over(window_drank_spec)
                                        ).filter(col('dense_rank')<=5)
                                       )
(dense_ranked_top_seller_products_df.select(
        'seller_id', 'product_id', 'price','total_orders', 
        format_number('total_revenue', 2).alias('total_revenue'), 
        'dense_rank'
    )
).show(truncate=False)

+--------------------------------+--------------------------------+------+------------+-------------+----------+
|seller_id                       |product_id                      |price |total_orders|total_revenue|dense_rank|
+--------------------------------+--------------------------------+------+------------+-------------+----------+
|001cca7ae9ae17fb1caed9dfb1094831|08574b074924071f4e201e151b152b4e|99.0  |60          |5,940.00     |1         |
|001cca7ae9ae17fb1caed9dfb1094831|08574b074924071f4e201e151b152b4e|89.0  |25          |2,225.00     |2         |
|001cca7ae9ae17fb1caed9dfb1094831|e251ebd2858be1aa7d9b2087a6992580|112.0 |18          |2,016.00     |3         |
|001cca7ae9ae17fb1caed9dfb1094831|547b95702aec86f05ac37e61d164891c|129.0 |9           |1,161.00     |4         |
|001cca7ae9ae17fb1caed9dfb1094831|0da9ffd92214425d880de3f94e74ce39|112.0 |10          |1,120.00     |5         |
|001e6ad469a905060d959994f1b41e4f|093cd981b714bcdff182b427d87fc8fc|250.0 |1           |250.00   

## Advanced Aggregation

### Total Revenue & Average Order Value (AOV) per Customer

In [16]:
customer_spending_df = (full_orders_df
    .groupBy('customer_unique_id')
    .agg(
        count('order_id').alias('total_orders'),
        sum('payment_value').alias('revenue'),
    )
    .withColumn('AOV', round(col('revenue') / col('total_orders'), 2))
    .orderBy(desc('revenue'))
)

customer_spending_df.select('customer_unique_id', 'total_orders', format_number('revenue',2).alias('revenue'), 'AOV').show()

+--------------------+------------+--------+-------+
|  customer_unique_id|total_orders| revenue|    AOV|
+--------------------+------------+--------+-------+
|c8460e4251689ba20...|           4|4,655.91|1163.98|
|adfa1cab2b2c8706d...|           1|3,242.84|3242.84|
|1b76903617af13189...|           1|3,195.73|3195.73|
|be825ddd3b40db3f9...|           1|3,122.72|3122.72|
|ef8d54b3797ea4db1...|           1|3,018.60| 3018.6|
|fff5eb4918b2bf4b2...|           1|2,844.96|2844.96|
|eae0a83d752b1dd32...|           2|2,783.01|1391.51|
|26b5839e505ee33e6...|           1|2,607.09|2607.09|
|94c1c602b16a9fdfb...|           1|2,566.90| 2566.9|
|575fa0cb05146c256...|           1|2,480.58|2480.58|
|bef3ce10ea5715fbc...|           1|2,467.17|2467.17|
|86df00dc5fd68f4dd...|           3|2,400.48| 800.16|
|559026a1299bd2ede...|           1|2,276.10| 2276.1|
|698e1cf81d01a3d38...|           1|2,262.80| 2262.8|
|3d47f4368ccc8e1bb...|           1|2,234.66|2234.66|
|3895f60f6e6a89e5c...|           1|2,204.04|22

### Seller Performance Metrics (Revenue, Average Review Score, Order Count)

In [17]:
seller_performance_df = (full_orders_df.groupBy('seller_id')
                     .agg(
                         count('order_id').alias('total_orders'),
                         sum('price').alias('total_revenue'),
                         round(avg('review_score'),2).alias('avg_review_score'),
                         round(stddev('price'),2).alias('price_variability')
                     )
                     .orderBy(desc('total_revenue'))
                     )

seller_performance_df.select('seller_id', format_number('total_orders',2).alias('total_orders'), 
                             format_number('total_revenue',2).alias('total_revenue'), 'avg_review_score',
                            'price_variability').show()

+--------------------+------------+-------------+----------------+-----------------+
|           seller_id|total_orders|total_revenue|avg_review_score|price_variability|
+--------------------+------------+-------------+----------------+-----------------+
|4869f7a5dfa277a7d...|    1,127.00|   223,980.16|            4.14|           121.08|
|4a3ca9315b744ce9f...|    1,749.00|   177,763.07|            3.85|            60.75|
|fa1c13f2614d7b5c4...|      555.00|   161,330.25|            4.34|           184.78|
|7a67c85e85bb2ce85...|    1,145.00|   139,659.79|            4.27|            54.02|
|7c67e1448b00f6e96...|      976.00|   138,365.41|            3.49|             46.7|
|da8622b14eb17ae28...|    1,292.00|   132,148.19|            4.19|             68.1|
|7e93a43ef30c4f03f...|      291.00|   119,554.30|            4.26|           231.27|
|46dc3b2cc0980fb8e...|      513.00|   116,741.44|            4.21|           126.42|
|955fee9216a65b617...|    1,280.00|   115,986.23|            4.17

### Product Popularity Metrics

In [18]:
product_metrics_df = (full_orders_df.groupBy('product_id')
                         .agg(
                             count('order_id').alias('total_sales'),
                             sum('price').alias('total_revenue'),
                             round(avg('price'),2).alias('avg_price'),
                             round(stddev('price'), 2).alias('price_volatility'),
                             collect_set('seller_id').alias('unique_sellers'),
                             countDistinct('seller_id').alias('unique_seller_count')
                         ).orderBy(desc('total_sales'))
                     )
product_metrics_df.show()

+--------------------+-----------+------------------+---------+----------------+--------------------+-------------------+
|          product_id|total_sales|     total_revenue|avg_price|price_volatility|      unique_sellers|unique_seller_count|
+--------------------+-----------+------------------+---------+----------------+--------------------+-------------------+
|99a4788cb24856965...|        438|          38611.36|    88.15|            3.46|[4a3ca9315b744ce9...|                  2|
|aca2eb7d00ea1a7b8...|        430|30660.699999999997|     71.3|            3.81|[955fee9216a65b61...|                  1|
|422879e10f4668299...|        337|18440.300000000003|    54.72|            4.42|[1f50f920176fa81d...|                  1|
|d1c427060a0f73f6b...|        314| 43118.66000000001|   137.32|           16.85|[a1043bafd471dff5...|                  1|
|53b36df67ebb7c415...|        302|35140.020000000004|   116.36|           19.69|[7d13fca152253586...|                  3|
|389d119b48cf3043d...|  

### Monthly Revenue And Order Count

In [19]:
monthly_stat_df = (full_orders_df
    .groupBy('purchase_month_name', 'purchase_month', 'purchase_year')
    .agg(
        sum('price').alias('seller_revenue'),
        sum('payment_value').alias('customer_spend'),
        sum('freight_value').alias('total_freight'),
        count('order_id').alias('total_orders'),
        min('price').alias('min_order_value'),
        max('price').alias('max_order_value')
    )
    .withColumn('aov_seller', round(col('seller_revenue') / col('total_orders'), 2))
    .withColumn('aov_customer', round(col('customer_spend') / col('total_orders'), 2))
    .orderBy('purchase_year', 'purchase_month', desc('seller_revenue'))  
)

monthly_stat_df.select(
    'purchase_month_name',
    'purchase_year',
    format_number('seller_revenue', 2).alias('seller_revenue'),
    format_number('customer_spend', 2).alias('customer_spend'),
    format_number('total_freight', 2).alias('total_freight'),
    format_number('total_orders', 0).alias('total_orders'),
    format_number('aov_seller', 2).alias('aov_seller'),      
    format_number('aov_customer', 2).alias('aov_customer'),  
    format_number('min_order_value', 2).alias('min_order_value'),
    format_number('max_order_value', 2).alias('max_order_value')
).show()

+-------------------+-------------+--------------+--------------+-------------+------------+----------+------------+---------------+---------------+
|purchase_month_name|purchase_year|seller_revenue|customer_spend|total_freight|total_orders|aov_seller|aov_customer|min_order_value|max_order_value|
+-------------------+-------------+--------------+--------------+-------------+------------+----------+------------+---------------+---------------+
|                Sep|         2016|        144.48|        211.29|        50.06|           3|     48.16|       70.43|          39.99|          59.50|
|                Oct|         2016|     38,470.29|     49,288.36|     6,017.72|         300|    128.23|      164.29|          10.00|         739.98|
|                Dec|         2016|         10.90|         19.62|         8.72|           1|     10.90|       19.62|          10.90|          10.90|
|                Jan|         2017|     96,147.29|    122,472.66|    14,411.21|         761|    126.34|   

### Customer Retention Rate

In [20]:
total_order = full_orders_df.count()

customer_retention_df = (full_orders_df
    .groupBy('customer_unique_id')
    .agg(
        min('order_purchase_timestamp').alias('first_order_date'),
        max('order_purchase_timestamp').alias('last_order_date'),
        countDistinct('order_id').alias('total_orders'),
        sum('payment_value').alias('total_amount_spent'),
    )
    .withColumn('aov', round(col('total_amount_spent') / col('total_orders'), 2))
    .withColumn('retention_rate', round((col('total_orders') / lit(total_order)) * 100, 2))
    .orderBy('first_order_date', 'last_order_date', desc('total_amount_spent'), desc('retention_rate'))
)

customer_retention_df.select(
    'customer_unique_id',
    'first_order_date',
    'last_order_date',
    format_number('total_orders', 0).alias('total_orders'),
    format_number('total_amount_spent', 2).alias('total_amount_spent'),
    format_number('aov', 2).alias('aov'),
    format_number('retention_rate', 2).alias('retention_rate')
).orderBy(desc('retention_rate')).show()

+--------------------+-------------------+-------------------+------------+------------------+------+--------------+
|  customer_unique_id|   first_order_date|    last_order_date|total_orders|total_amount_spent|   aov|retention_rate|
+--------------------+-------------------+-------------------+------------+------------------+------+--------------+
|8d50f5eadf50201cc...|2017-05-15 23:30:03|2018-08-20 19:14:26|          16|            902.04| 56.38|          0.02|
|b4e4f24de1e8725b7...|2018-02-23 13:12:21|2018-03-30 20:48:54|           5|            274.95| 54.99|          0.01|
|f0e310a6839dce9de...|2017-05-20 08:53:30|2018-04-05 09:04:45|           6|            540.69| 90.12|          0.01|
|56c8638e7c058b98a...|2017-07-12 06:48:11|2018-04-19 18:54:32|           5|            806.47|161.29|          0.01|
|47c1a3033b8b77b3a...|2017-08-07 14:14:22|2018-01-24 15:15:26|           6|            944.21|157.37|          0.01|
|74cb1ad7e6d567432...|2017-08-28 20:24:52|2018-04-03 01:16:22|  

### Data Enrichment

#### Order Status Flag

In [21]:
statuses = [row['order_status'] for row in full_orders_df.select('order_status').distinct().collect()]

for status in statuses:
    full_orders_df = full_orders_df.withColumn(
        f'is_{status}',
        when(col('order_status') == status, True).otherwise(False)
    )
full_orders_df.select('order_id', *[f'is_{status}' for status in statuses]).show()

+--------------------+-------------+------------+-----------+----------+-----------+--------------+-----------+
|            order_id|is_processing|is_delivered|is_invoiced|is_shipped|is_canceled|is_unavailable|is_approved|
+--------------------+-------------+------------+-----------+----------+-----------+--------------+-----------+
|4c5fb984844ad48dc...|        false|        true|      false|     false|      false|         false|      false|
|7828e49a0881db0de...|        false|        true|      false|     false|      false|         false|      false|
|1fc4c658ba7a1f8ed...|        false|        true|      false|     false|      false|         false|      false|
|b19b783be83450d0c...|        false|        true|      false|     false|      false|         false|      false|
|7d32c87acba91ed87...|        false|        true|      false|     false|      false|         false|      false|
|cf55d16f64505cb07...|        false|        true|      false|     false|      false|         false|     

#### Customer Segmentation from their AOV

In [22]:
PREMIUM_CUSTOMER_SPEND_LIMIT = 1500
MEDIUM_CUSTOMER_SPEND_LIMIT = 700

customer_spending_df = (customer_spending_df
    .withColumn(
        'customer_type',
        when(col('AOV') > PREMIUM_CUSTOMER_SPEND_LIMIT, 'Premium')
        .when(col('AOV') > MEDIUM_CUSTOMER_SPEND_LIMIT, 'Medium')
        .otherwise('Standard')
    )
)

full_orders_df = full_orders_df.join(customer_spending_df.select('customer_unique_id','customer_type'), 'customer_unique_id', 'left')

full_orders_df.select('customer_unique_id','customer_type').show()

+--------------------+-------------+
|  customer_unique_id|customer_type|
+--------------------+-------------+
|002043098f10ba39a...|     Standard|
|0035029989e6fc5cf...|     Standard|
|0047f3e16441284d7...|     Standard|
|009062f98089a0916...|     Standard|
|00a39521eb40f7012...|     Standard|
|00db525fd8d4023dc...|     Standard|
|00dd4390a8e8ad7a1...|     Standard|
|01376b24e146a15f5...|     Standard|
|013f4353d26bb05dc...|     Standard|
|016201b260065cecb...|     Standard|
|0255fc149c1f19131...|     Standard|
|026c7e2e8e363c7e7...|     Standard|
|02c7fc67a65e27a6a...|     Standard|
|02de4549062c86641...|     Standard|
|02fd7a082b1184535...|     Standard|
|0377c4b5792adc66c...|     Standard|
|0398b8575daf8b69a...|     Standard|
|03ec08f1dfebcf9f6...|     Standard|
|03f57ac1bec6632e9...|     Standard|
|04530097e2b38fa53...|     Standard|
+--------------------+-------------+
only showing top 20 rows



### Final Elimination of duplicates

In [23]:
full_orders_df = full_orders_df.dropDuplicates(['customer_unique_id', 'order_id', 'order_purchase_timestamp'])

### Preview Schemas for Analyzed Dataframe

#### Complete Order Details

In [24]:
getSchema(full_orders_df, "Complete Orders")

Schemas for Complete Orders
root
 |-- customer_unique_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- approval_time: integer (nullable = true)
 |-- carrier_handling_time: integer (nullable = true)
 |-- delivery_delay: integer (nullable = true)
 |-- delivery_duration: integer (nullable = true)
 |-- shipping_time: integer (nullable = true)
 |-- purchase_day: integer (nullable = true)
 |-- purchase_day_name: string (nullable = true)
 |-- purchase_m

#### Order Per Customers

In [25]:
getSchema(orders_per_customers_df, "Customers Orders")

Schemas for Customers Orders
root
 |-- customer_unique_id: string (nullable = true)
 |-- total_order: long (nullable = false)



#### Average Review Score for Sellers

In [26]:
getSchema(seller_avg_review_score_df, "Seller Review Score")

Schemas for Seller Review Score
root
 |-- seller_id: string (nullable = true)
 |-- avg_review_score: double (nullable = true)



#### Most Sold Products

In [27]:
getSchema(most_sold_products_df, "Most Sold products")

Schemas for Most Sold Products
root
 |-- product_id: string (nullable = true)
 |-- total_order: long (nullable = false)



#### Customers Revenue

In [28]:
getSchema(revenue_customers_df, "Customer Revenue")

Schemas for Customer Revenue
root
 |-- customer_id: string (nullable = true)
 |-- total_amount_spent: double (nullable = true)



#### Seller Products

In [29]:
getSchema(seller_products_df, "seller products")

Schemas for Seller Products
root
 |-- seller_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- price: double (nullable = true)
 |-- total_orders: long (nullable = false)
 |-- total_revenue: double (nullable = true)



#### Ranked Top Sellers

In [30]:
getSchema(ranked_top_seller_products_df, "Ranked Top Sellers")

Schemas for Ranked Top Sellers
root
 |-- seller_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- price: double (nullable = true)
 |-- total_orders: long (nullable = false)
 |-- total_revenue: double (nullable = true)
 |-- rank: integer (nullable = false)



#### Densed Ranked Top Customers

In [31]:
getSchema(dense_ranked_top_seller_products_df, "Densed Rnked Customers")

Schemas for Densed Rnked Customers
root
 |-- seller_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- price: double (nullable = true)
 |-- total_orders: long (nullable = false)
 |-- total_revenue: double (nullable = true)
 |-- dense_rank: integer (nullable = false)



#### Customer Spending Habit

In [32]:
getSchema(customer_spending_df, "Customer Spending Habit")

Schemas for Customer Spending Habit
root
 |-- customer_unique_id: string (nullable = true)
 |-- total_orders: long (nullable = false)
 |-- revenue: double (nullable = true)
 |-- AOV: double (nullable = true)
 |-- customer_type: string (nullable = false)



#### Seller Performance

In [33]:
getSchema(seller_performance_df, "seller performance")

Schemas for Seller Performance
root
 |-- seller_id: string (nullable = true)
 |-- total_orders: long (nullable = false)
 |-- total_revenue: double (nullable = true)
 |-- avg_review_score: double (nullable = true)
 |-- price_variability: double (nullable = true)



#### Product Metrics

In [34]:
getSchema(product_metrics_df, "product metrics")

Schemas for Product Metrics
root
 |-- product_id: string (nullable = true)
 |-- total_sales: long (nullable = false)
 |-- total_revenue: double (nullable = true)
 |-- avg_price: double (nullable = true)
 |-- price_volatility: double (nullable = true)
 |-- unique_sellers: array (nullable = false)
 |    |-- element: string (containsNull = false)
 |-- unique_seller_count: long (nullable = false)



#### Monthly Statistics

In [35]:
getSchema(monthly_stat_df, "monthly stat")

Schemas for Monthly Stat
root
 |-- purchase_month_name: string (nullable = true)
 |-- purchase_month: integer (nullable = true)
 |-- purchase_year: integer (nullable = true)
 |-- seller_revenue: double (nullable = true)
 |-- customer_spend: double (nullable = true)
 |-- total_freight: double (nullable = true)
 |-- total_orders: long (nullable = false)
 |-- min_order_value: double (nullable = true)
 |-- max_order_value: double (nullable = true)
 |-- aov_seller: double (nullable = true)
 |-- aov_customer: double (nullable = true)



#### Customer Retention

In [36]:
getSchema(customer_retention_df, "customer retention")

Schemas for Customer Retention
root
 |-- customer_unique_id: string (nullable = true)
 |-- first_order_date: timestamp (nullable = true)
 |-- last_order_date: timestamp (nullable = true)
 |-- total_orders: long (nullable = false)
 |-- total_amount_spent: double (nullable = true)
 |-- aov: double (nullable = true)
 |-- retention_rate: double (nullable = true)



### Storing Analysed Data

In [37]:
full_orders_df.write.mode('overwrite').parquet('/user/Marho/project/olist/analyzed/completeOrderdetails')
orders_per_customers_df.write.mode('overwrite').parquet('/user/Marho/project/olist/analyzed/customerOrders')
seller_avg_review_score_df.write.mode('overwrite').parquet('/user/Marho/project/olist/analyzed/avgSellerRevenueScore')
most_sold_products_df.write.mode('overwrite').parquet('/user/Marho/project/olist/analyzed/mostSoldProducts')
revenue_customers_df.write.mode('overwrite').parquet('/user/Marho/project/olist/analyzed/customerPurchase')
seller_products_df.write.mode('overwrite').parquet('/user/Marho/project/olist/analyzed/sellersProduct')
ranked_top_seller_products_df.write.mode('overwrite').parquet('/user/Marho/project/olist/analyzed/rankedSellers')
dense_ranked_top_seller_products_df.write.mode('overwrite').parquet('/user/Marho/project/olist/analyzed/denseRankedSellers')
customer_spending_df.write.mode('overwrite').parquet('/user/Marho/project/olist/analyzed/customerSpending')
seller_performance_df.write.mode('overwrite').parquet('/user/Marho/project/olist/analyzed/sellerPerformance')
product_metrics_df.write.mode('overwrite').parquet('/user/Marho/project/olist/analyzed/productsMetric')
monthly_stat_df.write.mode('overwrite').parquet('/user/Marho/project/olist/analyzed/monthlyStat')
customer_retention_df.write.mode('overwrite').parquet('/user/Marho/project/olist/analyzed/customerRetention')

### Storing Analysed Data as Tables on Hive

_Olist Database was pre-created on Hive_

#### Store data as table in Olist Database

In [38]:
full_orders_df.write.format('parquet').mode('overwrite').saveAsTable('olist.completeOrderdetails')
orders_per_customers_df.write.format('parquet').mode('overwrite').saveAsTable('olist.customerOrders')
seller_avg_review_score_df.write.format('parquet').mode('overwrite').saveAsTable('olist.avgSellerRevenueScore')
most_sold_products_df.write.format('parquet').mode('overwrite').saveAsTable('olist.mostSoldProducts')
revenue_customers_df.write.format('parquet').mode('overwrite').saveAsTable('olist.customerPurchase')
seller_products_df.write.format('parquet').mode('overwrite').saveAsTable('olist.sellersProduct')
ranked_top_seller_products_df.write.format('parquet').mode('overwrite').saveAsTable('olist.rankedSellers')
dense_ranked_top_seller_products_df.write.format('parquet').mode('overwrite').saveAsTable('olist.denseRankedSellers')
customer_spending_df.write.format('parquet').mode('overwrite').saveAsTable('olist.customerSpending')
seller_performance_df.write.format('parquet').mode('overwrite').saveAsTable('olist.sellerPerformance')
product_metrics_df.write.format('parquet').mode('overwrite').saveAsTable('olist.productsMetric')
monthly_stat_df.write.format('parquet').mode('overwrite').saveAsTable('olist.monthlyStat')
customer_retention_df.write.format('parquet').mode('overwrite').saveAsTable('olist.customerRetention')

ivysettings.xml file not found in HIVE_HOME or HIVE_CONF_DIR,/etc/hive/conf.dist/ivysettings.xml will be used
26/06/29 16:03:54 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.


### Clear cache

In [40]:
full_orders_df.unpersist()

### Stop Session

In [44]:
spark.stop()